In [ ]:
#!pip install pyomo
#!pip install highspy

In [1]:
# Import libraries
from pyomo.environ import Param, Set, ConcreteModel, Var, Constraint, NonNegativeReals, Objective, maximize, SolverFactory

# Despacho económico

En el despacho nos interesa maximizar el benficio social, es decir, atender la demanda mientras mínimo los costos de producción

BS = Valor de la atención de la demanda - Costo de produccion

## Sets - conjuntos

$g \in G = \{g1, g2, g3\} : \text{Generadores}$  
$d \in D = \{d1, d2, d3, d4\} : \text{Demanda}$

## Parámetros

$P^{max}_{g}$: Potencia máxima del generador

Curvas de valoración y de costo  
$c^{dem}_{d}$: curva de cantidades y precio de demada  
$c^{gen}_{g}$: curva de cantidades y precios de la generación

## Variablers

$L_{d}$: Demanda de la carga d  
$P_{g}$: Generación del generador g

## Ecuaciones

### Función objetivo

$\max \: \sum_{d} L_{d} \cdot c^{dem}_{d} - \sum_{g} P_{g} \cdot c^{gen}_{g} $

### Restricciones
 
Balance de potencia:  
$\sum_{d} L_{d} = \sum_{g} P_{g} \; : \lambda$

# Límite máximo de los generadores  
$ P_{g} \le P^{max}_{g} \; \forall g$

# Límite máximo de demanda
$ L_{d} \le D^{max}_{d} \; \forall d$

# Creación del modelo y conjunts

In [2]:
# Definición del modelo
m = ConcreteModel("Despacho uninodal")

# Definición de los conjuntos
m.GENERATORS = Set(initialize=["Hydro", "Coal", "Gas"])
m.DEMAND_BLOCKS = Set(initialize=["D1", "D2", "D3", "D4"])

# Carga de datos (puede ser de un DB, excel, u otros)

In [3]:
generator_capacity = {
    "Hydro": 100,
    "Coal": 80,
    "Gas": 120,
}

generator_cost = {
    "Hydro": 20,
    "Coal": 40,
    "Gas": 90,
}

demand_capacity = {
    "D1": 50,
    "D2": 50,
    "D3": 50,
    "D4": 50,
}

demand_price = {
    "D1": 150,
    "D2": 110,
    "D3": 70,
    "D4": 30,
}

In [4]:
# pasamos los parámetros al objeto de nuestro módelo, para poder llamarlo de manera simplificada
m.GenMax = Param(
    m.GENERATORS,
    initialize=generator_capacity
)

m.GenCost = Param(
    m.GENERATORS,
    initialize=generator_cost
)

m.DemandMax = Param(
    m.DEMAND_BLOCKS,
    initialize=demand_capacity
)

m.DemandPrice = Param(
    m.DEMAND_BLOCKS,
    initialize=demand_price
)

# Variables

In [5]:
# Generation dispatch
m.pg = Var(m.GENERATORS, domain=NonNegativeReals)

# Accepted demand
m.pd = Var(m.DEMAND_BLOCKS, domain=NonNegativeReals)

# Modelo
## Función objetivo

In [6]:
def social_wellfare(model):
  valor_demanda = sum([model.pd[d] * model.DemandPrice[d] for d in model.DEMAND_BLOCKS])
  costo_gen = sum([model.pg[g] * model.GenCost[g] for g in model.GENERATORS])
  return valor_demanda - costo_gen
m.fo = Objective(rule=social_wellfare, sense=maximize)

## Sujeto a:

In [7]:
# Balance de potencia
def balance(model):
  return sum([model.pg[g] for g in model.GENERATORS]) == sum([model.pd[d] for d in model.DEMAND_BLOCKS])
m.balance = Constraint(rule=balance, name='conservacion')

# Potencia máxima de los generadores
def max_gen(model, g):
  return model.pg[g] <= model.GenMax[g]
m.max_gen = Constraint(m.GENERATORS, rule=max_gen, name='max_gen')

# Conusmo máximo de los bloques de demanda
def max_demand(model, d):
  return model.pd[d] == model.DemandMax[d]
m.max_demand = Constraint(m.DEMAND_BLOCKS, rule=max_demand, name='max_demand')

# Solver

In [8]:
solver = SolverFactory("highs")
options = {"symbolic_solver_labels" : True}
m.write('despacho.lp', io_options = options)
results = solver.solve(m, tee=True)

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP has 8 rows; 7 cols; 14 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+01, 2e+02]
  Bound   [0e+00, 0e+00]
  RHS     [5e+01, 1e+02]
Presolving model
1 rows, 3 cols, 3 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-8); columns 0(-7); nonzeros 0(-14) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  1.1000000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [12]:
m.pprint()

2 Set Declarations
    DEMAND_BLOCKS : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'D1', 'D2', 'D3', 'D4'}
    GENERATORS : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'Hydro', 'Coal', 'Gas'}

4 Param Declarations
    DemandMax : Size=4, Index=DEMAND_BLOCKS, Domain=Any, Default=None, Mutable=False
        Key : Value
         D1 :    50
         D2 :    50
         D3 :    50
         D4 :    50
    DemandPrice : Size=4, Index=DEMAND_BLOCKS, Domain=Any, Default=None, Mutable=False
        Key : Value
         D1 :   150
         D2 :   110
         D3 :    70
         D4 :    30
    GenCost : Size=3, Index=GENERATORS, Domain=Any, Default=None, Mutable=False
        Key   : Value
         Coal :    40
          Gas :    90
        Hydro :    20
    GenMax : Size=3, Index=GENERATORS, Domain=Any, Default=None, Mutable=False
     

In [9]:
print(results)


Problem: 
- Lower bound: 11000.0
  Upper bound: 11000.0
  Number of objectives: 1
  Number of constraints: nan
  Number of variables: nan
  Sense: maximize
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.convergenceCriteriaSatisfied



In [11]:
m.pd.pprint()

pd : Size=4, Index=DEMAND_BLOCKS
    Key : Lower : Value : Upper : Fixed : Stale : Domain
     D1 :     0 :  50.0 :  None : False : False : NonNegativeReals
     D2 :     0 :  50.0 :  None : False : False : NonNegativeReals
     D3 :     0 :  50.0 :  None : False : False : NonNegativeReals
     D4 :     0 :  50.0 :  None : False : False : NonNegativeReals


In [13]:
m.pg.pprint()

pg : Size=3, Index=GENERATORS
    Key   : Lower : Value : Upper : Fixed : Stale : Domain
     Coal :     0 :  80.0 :  None : False : False : NonNegativeReals
      Gas :     0 :  20.0 :  None : False : False : NonNegativeReals
    Hydro :     0 : 100.0 :  None : False : False : NonNegativeReals
